In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv, find_dotenv

# 프로젝트 루트는 .env 파일이 있는 폴더 (어디서 실행해도 안전)
ENV_PATH = find_dotenv()
PROJECT_ROOT = Path(ENV_PATH).parent
load_dotenv(ENV_PATH)
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
assert OPENAI_API_KEY, ".env 파일에 OPENAI_API_KEY가 없습니다."

from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# 1. 아까 구워둔 FAISS DB 1초 만에 불러오기 (비용 0원, 속도 즉시!)
db_path = str(PROJECT_ROOT / "vector_db" / "faiss_stat501_db")
embeddings = OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY)

# allow_dangerous_deserialization=True 옵션은 내가 직접 만든 안전한 DB를 불러올 때 필수입니다.
vector_db = FAISS.load_local(db_path, embeddings, allow_dangerous_deserialization=True)

# 검색기 변환 (가장 관련성 높은 4개의 조각을 찾아오도록 세팅)
retriever = vector_db.as_retriever(search_kwargs={"k": 4})

# 2. 챗봇 두뇌 세팅 (온도 0으로 팩트 위주 설정)
llm = ChatOpenAI(model="gpt-4o-mini", openai_api_key=OPENAI_API_KEY, temperature=0)

# 3. 🛡️ 과적합을 방지한 완벽한 범용 마스터 프롬프트
prompt_template = """
너는 정보통계학과 학생들을 위한 친절하고 전문적인 통계 어시스턴트야.
반드시 아래에 제공된 [검색된 전공 자료]만을 근거로 답변해.

지시사항:
1. 검색된 영어 원문을 통계학 초보자도 이해하기 쉽게 자연스러운 한국어로 번역하고 풀어서 설명해.
2. 수식은 LaTeX 문법($ 기호)을 사용해서 깔끔하게 작성해.
3. 📊 [데이터 및 통계표 출력] 원문에 데이터 테이블, 분산분석표(ANOVA), 회귀계수표, 상관계수 등 어떠한 형태의 '표(Table) 데이터'가 존재한다면, 생략하지 말고 Markdown 표 형식으로 깔끔하게 그려서 설명해.
4. 📈 [시각 자료의 논리적 배치] 원문에 산점도, 잔차도, 히스토그램 등 '통계 그래프 이미지' 링크(`![...](/...)`)가 있다면, 네가 해당 그래프의 의미나 분석 결과를 설명하는 문장 바로 뒤에 자연스럽게 이미지를 삽입해서 시각적 근거로 제시해.
   - 단, 이미지 주소가 '/'로 시작하는 상대 경로라면, 반드시 앞에 'https://online.stat.psu.edu'를 붙여서 절대 경로(`![그래프이름](https://online.stat.psu.edu/...)`)로 변환해서 출력해.
5. 🛑 제공된 자료에 없는 내용은 절대 네 뇌피셜로 지어내지 마. 모르면 "제공된 가이드라인에서는 해당 내용을 찾을 수 없습니다"라고 답해.

[검색된 전공 자료]:
{context}

질문: {input}
"""

prompt = PromptTemplate.from_template(prompt_template)

# 4. LCEL 파이프라인 조립
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "input": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# =========================================================
# 🚀 실전 테스트: 1단원부터 15단원까지 아무거나 물어보세요!
# =========================================================

# 예시 1: 다중공선성 (12단원)
question_1 = "다중공선성이 문제가 되는 이유와 데이터 기반 다중공선성에 대해 설명해줘."

# 예시 2: 단순선형회귀 (1단원 - 새로운 데이터 검증!)
question_2 = "결정계수 R^2의 의미가 무엇인가요? 0일 때의 주의점은?"

print(f"질문 1: {question_1}")
print("🤖 답변:")
print(rag_chain.invoke(question_1))
print("-" * 50)

print(f"질문 2: {question_2}")
print("🤖 답변:")
print(rag_chain.invoke(question_2))

질문 1: 다중공선성이 문제가 되는 이유와 데이터 기반 다중공선성에 대해 설명해줘.
🤖 답변:
다중공선성은 회귀 분석에서 두 개 이상의 예측 변수가 서로 강하게 상관관계를 가질 때 발생하는 현상입니다. 이로 인해 회귀 계수의 추정치가 불안정해지고, 분석 결과의 해석이 어려워질 수 있습니다. 다중공선성이 문제가 되는 이유는 다음과 같습니다:

1. **회귀 계수의 의존성**: 특정 변수의 회귀 계수는 모델에 포함된 다른 예측 변수에 따라 달라질 수 있습니다. 즉, 어떤 변수를 포함하느냐에 따라 그 변수의 중요성이 달라질 수 있습니다.

2. **정확도 감소**: 예측 변수를 추가할수록 회귀 계수의 추정 정확도가 떨어질 수 있습니다. 이는 다중공선성으로 인해 계수의 분산이 증가하기 때문입니다.

3. **한계 기여도**: 특정 예측 변수가 오차 제곱합을 줄이는 데 기여하는 정도는 다른 예측 변수가 이미 모델에 포함되어 있는지에 따라 달라질 수 있습니다.

4. **가설 검정의 불확실성**: 특정 회귀 계수가 0이라는 가설 검정 결과가 모델에 포함된 예측 변수에 따라 달라질 수 있습니다. 이는 연구 결과의 신뢰성을 떨어뜨립니다.

이러한 이유로 다중공선성을 줄이는 것이 중요합니다. 다중공선성이 존재하면 회귀 계수의 신뢰 구간이 넓어져 유용성이 떨어지기 때문에, 이를 줄이면 더 좁고 유용한 신뢰 구간을 얻을 수 있습니다.

### 데이터 기반 다중공선성

데이터 기반 다중공선성은 실험 설계가 잘못되었거나, 관찰 데이터에만 의존하거나, 데이터 수집 시스템을 조작할 수 없는 경우에 발생합니다. 이는 구조적 다중공선성과는 달리, 연구자가 의도하지 않은 방식으로 발생하는 문제입니다. 데이터 기반 다중공선성은 일반적으로 더 심각한 문제로, 연구에서 자주 발생합니다.

예를 들어, 고혈압 환자 20명의 데이터를 수집한 연구에서 다음과 같은 변수들이 관찰되었습니다:

- 혈압 (*y* = *BP*, mm Hg)
- 나이 (\(x_1 = \text{Age}\), 년)
- 체중 (\